In [1]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'refactor-1'

# Install lightweight dependencies needed for smoke checks.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', '-q'],
    check=True,
)

# Clone or update exact branch used for current development.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

Repo ready: /content/Yolo-ST-GCN
Branch: refactor-1


In [2]:
# !git pull

In [3]:
!python scripts/build_gym99_from_gym288.py --experiment_config configs/experiments/gym99_build_from_gym288.json

usage: build_gym99_from_gym288.py [-h] [--experiment_config EXPERIMENT_CONFIG]
                                  --gym288_dataset_path GYM288_DATASET_PATH
                                  --gym99_dataset_path GYM99_DATASET_PATH
                                  [--gym288_categories_url GYM288_CATEGORIES_URL]
                                  [--gym99_categories_url GYM99_CATEGORIES_URL]
                                  [--disable_neighbor_fallback]
build_gym99_from_gym288.py: error: the following arguments are required: --gym288_dataset_path, --gym99_dataset_path


In [4]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import snapshot_download

download_dir = Path('/content/Gym288-skeleton')
if not (download_dir.exists() and any(download_dir.iterdir())):
    print('Downloading Gym288-skeleton...')
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
else:
    print('Gym288-skeleton already exists.')

pkl_candidates = sorted(download_dir.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('No .pkl file found for Gym288 dataset.')

gym288_pkl = str(pkl_candidates[0])
print('Gym288 pickle:', gym288_pkl)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Gym288 pickle: /content/Gym288-skeleton/gym_288_skeleton.pkl


In [5]:
# !python scripts/train_gym99.py \
# --auto_build_from_gym288 \
# --gym288_dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
# --dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl \
# --out_dir outputs/refactor1_2ep/gym99_coco18_2s \
# --epochs 2 \
# --batch_size 256 \
# --lr 0.001 \
# --num_workers 0 \
# --joint_spec_name coco18 \
# --use_two_stream

Building Gym99-from-Gym288 pickle...
Gym99 mapping stats: direct=34240 minus1=1625 plus1=800 train=27624 test=9041
Device: cuda
Loading Gym99-skeleton dataset...
Traceback (most recent call last):
  File "/content/Yolo-ST-GCN/scripts/train_gym99.py", line 294, in <module>
  File "/content/Yolo-ST-GCN/scripts/train_gym99.py", line 122, in main
    data, bone_data, labels, flags, _, _ = build_gym99_data_tensors(
                                           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Yolo-ST-GCN/src/gym99_dataset.py", line 125, in build_gym99_data_tensors
    all_bone_data.append(calculate_bone_data(tensor, bone_pairs).astype(np.float32))
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Yolo-ST-GCN/src/skeleton_utils.py", line 194, in calculate_bone_data
    bone_data[:, :, child_idx, :] = arr[:, :, child_idx, :] - arr[:, :, parent_idx, :]
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C


In [8]:
!git pull

remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 12 (delta 10), reused 12 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.63 KiB | 539.00 KiB/s, done.
From https://github.com/schizocatto/Yolo-ST-GCN
   a6b9043..4e3e40d  refactor-1 -> origin/refactor-1
Updating a6b9043..4e3e40d
Fast-forward
 scripts/inference_gym288.py |  2 +-
 scripts/inference_gym99.py  |  2 +-
 scripts/train.py            | 25 +++++++++++++++++++++++++
 scripts/train_gym288.py     | 26 ++++++++++++++++++++++++++
 scripts/train_gym99.py      | 30 ++++++++++++++++++++++++++++++
 src/inference.py            |  2 +-
 src/model.py                | 34 +++++++++++++++++++++++-----------
 src/train.py                | 12 +++++++++++-
 8 files changed, 118 insertions(+), 15 deletions(-)


In [ ]:
!python scripts/train_gym99.py \
--auto_build_from_gym288 \
--gym288_dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
--dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl \
--out_dir outputs/refactor1_2ep/gym99_coco18_2s \
--epochs 30 \
--batch_size 256 \
--lr 0.001 \
--num_workers 0 \
# --max_train_samples 8192 \
# --max_val_samples 2048 \
--joint_spec_name coco18 \
--save_every_epochs 10 \
--use_two_stream
# --train_data_mode preload_vram

Building Gym99-from-Gym288 pickle...
Gym99 mapping stats: direct=34240 minus1=1625 plus1=800 train=27624 test=9041
Device: cuda
Loading Gym99-skeleton dataset...
Loaded 36665 samples  train=8192  test=2048
num_classes=99 (inferred=99)
DataLoader num_workers=0  two_stream=True  train_data_mode=standard
Epoch 1/30  train_loss=3.4929  train_acc=0.1685  val_loss=3.2351  val_acc=0.2744  val_f1=0.0504
Epoch 2/30  train_loss=2.5142  train_acc=0.3281  val_loss=2.2954  val_acc=0.4028  val_f1=0.1182
Epoch 3/30  train_loss=1.9859  train_acc=0.4329  val_loss=1.9320  val_acc=0.4434  val_f1=0.1778
Epoch 4/30  train_loss=1.6525  train_acc=0.4943  val_loss=1.7323  val_acc=0.4761  val_f1=0.2130
Epoch 5/30  train_loss=1.4276  train_acc=0.5566  val_loss=1.4430  val_acc=0.5522  val_f1=0.2787
Saved periodic checkpoint: outputs/refactor1_2ep/gym99_coco18_2s/stgcn_gym99_coco18_2s_epoch5.pth
Epoch 6/30  train_loss=1.2844  train_acc=0.5884  val_loss=1.3459  val_acc=0.5820  val_f1=0.3154
Epoch 7/30  train_loss=